# 第二题：预测分位数下的购电策略对比

本 notebook 读取 `q2/results/predict_with_price.xlsx`，将已完成的负载与光伏预测转为净负载策略，并参考第一问的储能调度约束比较四类结果：

- **保守策略**：使用 `净负载p95` 制定计划，尽量避免实际执行时出现紧急购电。
- **中庸策略**：使用 `净负载p50`，即中心预测。
- **激进策略**：使用 `净负载p05`，按乐观净负载制定计划，允许更高紧急购电风险。
- **实际最优**：提前知道 2025-02-01 至 2025-12-31 的实际净负载，对整个评价期统一优化，作为理想下界。

单位口径与第一问保持一致：功率 kW 乘以 10 分钟得到区间电量 kWh，储能充放电效率各取 90%，容量为 1200 至 10800 kWh，最大充放电功率均为 5000 kW。三种预测策略先用预测净负载制定当天完整计划（普通购电、充电、放电、储能轨迹），再将该计划放到实际净负载上执行；实际净负载偏高的缺口记为紧急购电，偏低的余量记为未利用电量。普通购电按电价结算，紧急购电按 5 倍电价结算。

## 1. 导入库和参数

In [1]:
from pathlib import Path
import json
import math
import time
import warnings

import numpy as np
import pandas as pd
from scipy.optimize import Bounds, LinearConstraint, linprog, milp
from scipy.sparse import coo_matrix, csr_matrix, hstack

try:
    from IPython.display import display
except ModuleNotFoundError:
    display = print

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

ROOT = Path.cwd()
while not (ROOT / "C题.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

PREDICT_XLSX = ROOT / "q2" / "results" / "predict_with_price.xlsx"

CFG = {
    "interval_minutes": 10,
    "charge_efficiency": 0.9,
    "discharge_efficiency": 0.9,
    "storage_min_kwh": 1200.0,
    "storage_max_kwh": 10800.0,
    "initial_storage_kwh": 6000.0,
    "terminal_storage_kwh": 6000.0,
    "max_charge_power_kw": 5000.0,
    "max_discharge_power_kw": 5000.0,
    "emergency_multiplier": 5.0,
    "tolerance": 1e-6,
}

STRATEGIES = {
    "保守": {"column": "净负载p95", "description": "高净负载分位数，尽量少发生紧急购电"},
    "中庸": {"column": "净负载p50", "description": "中心预测"},
    "激进": {"column": "净负载p05", "description": "低净负载分位数，乐观少买普通电"},
}

## 2. 读取预测工作簿

In [2]:
def load_predict_workbook(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)

    xls = pd.ExcelFile(path)
    frames = []
    for sheet in xls.sheet_names:
        df = pd.read_excel(path, sheet_name=sheet)
        required = ["时间", "电价", "实际净负载", "净负载p05", "净负载p50", "净负载p95"]
        missing = [c for c in required if c not in df.columns]
        if missing:
            raise KeyError(f"{sheet} 缺少列: {missing}")
        if len(df) != 144:
            raise ValueError(f"{sheet} 行数为 {len(df)}，期望 144")
        part = df[required].copy()
        part.insert(0, "date", pd.Timestamp(sheet))
        part.insert(1, "slot", np.arange(1, 145))
        frames.append(part)

    out = pd.concat(frames, ignore_index=True)
    numeric_cols = ["电价", "实际净负载", "净负载p05", "净负载p50", "净负载p95"]
    out[numeric_cols] = out[numeric_cols].apply(pd.to_numeric, errors="raise")
    if not np.isfinite(out[numeric_cols].to_numpy(float)).all():
        raise ValueError("预测工作簿存在非有限数值")
    if (out["电价"] <= 0).any():
        raise ValueError("电价必须为正")
    return out


raw = load_predict_workbook(PREDICT_XLSX)
dates = pd.DatetimeIndex(raw["date"].drop_duplicates())
prices = raw[raw["date"].eq(dates[0])]["电价"].to_numpy(float)

# kW -> kWh。净负载允许为负，表示光伏高于负载。
actual_net = raw["实际净负载"].to_numpy(float).reshape(len(dates), 144) / 6.0
forecast_paths = {
    name: raw[spec["column"]].to_numpy(float).reshape(len(dates), 144) / 6.0
    for name, spec in STRATEGIES.items()
}

print(f"读取 {len(dates)} 天，每天 144 个十分钟时段")
print(f"日期范围：{dates[0].date()} 至 {dates[-1].date()}")
display(raw.head())

读取 334 天，每天 144 个十分钟时段
日期范围：2025-02-01 至 2025-12-31


,date,slot,时间,电价,实际净负载,净负载p05,净负载p50,净负载p95
0,2025-02-01,1,00:10,0.4248,2532.1822,2076.370930,2674.520870,3144.648767
1,2025-02-01,2,00:20,0.4245,2492.4439,2101.604067,2730.463585,3050.043100
2,2025-02-01,3,00:30,0.4269,2464.3000,2282.965098,2821.635197,3120.558488
3,2025-02-01,4,00:40,0.4272,2581.7688,2496.361922,2834.485957,3132.598500
4,2025-02-01,5,00:50,0.4271,2546.4635,2434.174699,2850.937388,3216.728424


## 3. 调度优化与实际执行函数

In [3]:
def require(condition, message):
    if not bool(condition):
        raise ValueError(message)


def solve_dispatch_for_net(net_kwh, prices, initial_kwh, terminal_kwh, cfg=CFG, solver="auto"):
    """给定一条净负载曲线，求普通购电与储能调度的最小购电费。

    变量含义：
    g: 普通购电量；c: 充电量；d: 放电量；u: 未利用电量；E: 储能电量。
    约束为 g + d - c - u = net_load。
    """
    net = np.asarray(net_kwh, dtype=float)
    p = np.asarray(prices, dtype=float)
    n = len(net)
    require(n == len(p), "净负载和价格长度不一致")
    require(np.isfinite(net).all() and np.isfinite(p).all() and (p > 0).all(), "输入存在非法数值")

    eta_c = cfg["charge_efficiency"]
    eta_d = cfg["discharge_efficiency"]
    mc = cfg["max_charge_power_kw"] * cfg["interval_minutes"] / 60
    md = cfg["max_discharge_power_kw"] * cfg["interval_minutes"] / 60
    lo = cfg["storage_min_kwh"]
    hi = cfg["storage_max_kwh"]

    # x = [g(n), c(n), d(n), u(n), E(n+1)]
    count = 4 * n + n + 1
    obj = np.zeros(count)
    obj[:n] = p

    rows = []
    cols = []
    data = []
    rhs = np.zeros(2 * n)
    t = np.arange(n)

    # g + d - c - u = net
    rows.extend(t); cols.extend(t); data.extend(np.ones(n))
    rows.extend(t); cols.extend(2 * n + t); data.extend(np.ones(n))
    rows.extend(t); cols.extend(n + t); data.extend(-np.ones(n))
    rows.extend(t); cols.extend(3 * n + t); data.extend(-np.ones(n))
    rhs[:n] = net

    # E[t+1] - E[t] - eta_c*c + d/eta_d = 0
    r = n + t
    rows.extend(r); cols.extend(4 * n + t + 1); data.extend(np.ones(n))
    rows.extend(r); cols.extend(4 * n + t); data.extend(-np.ones(n))
    rows.extend(r); cols.extend(n + t); data.extend(-eta_c * np.ones(n))
    rows.extend(r); cols.extend(2 * n + t); data.extend(np.ones(n) / eta_d)

    eq = coo_matrix((data, (rows, cols)), shape=(2 * n, count)).tocsr()

    lower = np.zeros(count)
    upper = np.full(count, np.inf)
    upper[n:2*n] = mc
    upper[2*n:3*n] = md
    lower[4*n:] = lo
    upper[4*n:] = hi
    lower[4*n] = upper[4*n] = initial_kwh
    lower[5*n] = upper[5*n] = terminal_kwh

    lp = linprog(obj, A_eq=eq, b_eq=rhs, bounds=np.column_stack([lower, upper]), method="highs")
    if not lp.success:
        raise RuntimeError("LP failed: " + lp.message)

    x = lp.x
    used_solver = "HiGHS LP"
    overlap = (x[n:2*n] > cfg["tolerance"]) & (x[2*n:3*n] > cfg["tolerance"])
    if solver == "milp" or overlap.any():
        mode_rows = np.r_[t, t, n + t, n + t]
        mode_cols = np.r_[n + t, count + t, 2*n + t, count + t]
        mode_data = np.r_[np.ones(n), -mc * np.ones(n), np.ones(n), md * np.ones(n)]
        mode = coo_matrix((mode_data, (mode_rows, mode_cols)), shape=(2*n, count+n)).tocsr()
        res = milp(
            np.r_[obj, np.zeros(n)],
            integrality=np.r_[np.zeros(count), np.ones(n)],
            bounds=Bounds(np.r_[lower, np.zeros(n)], np.r_[upper, np.ones(n)]),
            constraints=[
                LinearConstraint(hstack([eq, csr_matrix((2*n, n))]), rhs, rhs),
                LinearConstraint(mode, -np.inf, np.r_[np.zeros(n), md*np.ones(n)]),
            ],
            options={"mip_rel_gap": 1e-8},
        )
        if not res.success:
            raise RuntimeError("MILP failed: " + res.message)
        x = res.x[:count]
        used_solver = "HiGHS MILP"

    residual = float(np.max(np.abs(eq @ x - rhs)))
    require(residual <= 1e-5, f"调度等式残差过大: {residual}")
    return {
        "grid": np.maximum(x[:n], 0),
        "charge": np.maximum(x[n:2*n], 0),
        "discharge": np.maximum(x[2*n:3*n], 0),
        "unused": np.maximum(x[3*n:4*n], 0),
        "storage": x[4*n:],
        "cost": float(p @ np.maximum(x[:n], 0)),
        "solver": used_solver,
        "residual": residual,
    }


def execute_plan_on_actual(plan, actual_net_kwh, initial_kwh, cfg=CFG):
    """严格按日前计划执行，再用实际净负载计算偏差。

    计划已经给出 grid、charge、discharge 和 storage 轨迹。
    当实际净负载高于计划可供给时，差额记为紧急购电；
    当实际净负载低于计划可供给时，差额记为未利用电量。
    这里不再根据当天真实值临场调整电池充放电。
    """
    grid = np.asarray(plan["grid"], dtype=float)
    charge = np.asarray(plan["charge"], dtype=float)
    discharge = np.asarray(plan["discharge"], dtype=float)
    storage = np.asarray(plan["storage"], dtype=float)
    net = np.asarray(actual_net_kwh, dtype=float)

    require(len(grid) == len(net) == len(charge) == len(discharge), "计划和实际曲线长度不一致")
    require(abs(storage[0] - initial_kwh) <= 1e-6, "计划期初储能和实际期初储能不一致")
    require(storage.min() >= cfg["storage_min_kwh"] - 1e-6 and storage.max() <= cfg["storage_max_kwh"] + 1e-6, "计划储能越界")
    require((charge <= cfg["max_charge_power_kw"] * cfg["interval_minutes"] / 60 + 1e-6).all(), "计划充电功率越界")
    require((discharge <= cfg["max_discharge_power_kw"] * cfg["interval_minutes"] / 60 + 1e-6).all(), "计划放电功率越界")

    planned_supply_after_battery = grid + discharge - charge
    emergency = np.maximum(net - planned_supply_after_battery, 0.0)
    unused = np.maximum(planned_supply_after_battery - net, 0.0)
    balance = grid + discharge + emergency - net - charge - unused
    require(np.max(np.abs(balance)) <= 1e-6, "实际执行能量平衡失败")

    return {
        "charge": charge,
        "discharge": discharge,
        "emergency": emergency,
        "unused": unused,
        "storage": storage,
    }


## 4. 三种预测策略回放

In [4]:
def run_forecast_strategy(name, forecast_net, actual_net, prices, cfg=CFG, progress=True):
    state = cfg["initial_storage_kwh"]
    day_rows = []
    slot_frames = []
    solver_counts = {}
    started = time.perf_counter()

    for k, date in enumerate(dates):
        plan = solve_dispatch_for_net(
            forecast_net[k],
            prices,
            initial_kwh=state,
            terminal_kwh=cfg["terminal_storage_kwh"],
            cfg=cfg,
        )
        actual = execute_plan_on_actual(plan, actual_net[k], state, cfg)
        solver_counts[plan["solver"]] = solver_counts.get(plan["solver"], 0) + 1

        planned_cost = float(prices @ plan["grid"])
        emergency_cost = float(cfg["emergency_multiplier"] * prices @ actual["emergency"])
        day_rows.append({
            "策略": name,
            "date": date,
            "计划购电量(kWh)": float(plan["grid"].sum()),
            "紧急购电量(kWh)": float(actual["emergency"].sum()),
            "未利用电量(kWh)": float(actual["unused"].sum()),
            "计划费用(元)": planned_cost,
            "紧急费用(元)": emergency_cost,
            "总费用(元)": planned_cost + emergency_cost,
            "期初储能(kWh)": state,
            "期末储能(kWh)": float(actual["storage"][-1]),
            "求解器": plan["solver"],
        })

        slot_frames.append(pd.DataFrame({
            "策略": name,
            "date": date,
            "slot": np.arange(1, 145),
            "actual_net_kwh": actual_net[k],
            "forecast_net_kwh": forecast_net[k],
            "planned_grid_kwh": plan["grid"],
            "planned_charge_kwh": plan["charge"],
            "planned_discharge_kwh": plan["discharge"],
            "emergency_kwh": actual["emergency"],
            "unused_kwh": actual["unused"],
            "storage_start_kwh": actual["storage"][:-1],
            "storage_end_kwh": actual["storage"][1:],
            "price_yuan_per_kwh": prices,
        }))

        state = float(actual["storage"][-1])
        if progress and (k + 1) % 50 == 0:
            print(f"{name}: 已完成 {k + 1}/{len(dates)} 天")

    print(f"{name}: 完成，用时 {time.perf_counter() - started:.1f}s，求解器统计 {solver_counts}")
    return pd.DataFrame(day_rows), pd.concat(slot_frames, ignore_index=True)


strategy_daily = []
strategy_detail = []

for strategy_name, path in forecast_paths.items():
    daily, detail = run_forecast_strategy(strategy_name, path, actual_net, prices)
    strategy_daily.append(daily)
    strategy_detail.append(detail)

daily_results = pd.concat(strategy_daily, ignore_index=True)
detail_results = pd.concat(strategy_detail, ignore_index=True)
display(daily_results.head())

保守: 已完成 50/334 天
保守: 已完成 100/334 天
保守: 已完成 150/334 天
保守: 已完成 200/334 天
保守: 已完成 250/334 天
保守: 已完成 300/334 天
保守: 完成，用时 1.0s，求解器统计 {'HiGHS LP': 334}
中庸: 已完成 50/334 天
中庸: 已完成 100/334 天
中庸: 已完成 150/334 天
中庸: 已完成 200/334 天
中庸: 已完成 250/334 天
中庸: 已完成 300/334 天
中庸: 完成，用时 1.0s，求解器统计 {'HiGHS LP': 334}
激进: 已完成 50/334 天
激进: 已完成 100/334 天
激进: 已完成 150/334 天
激进: 已完成 200/334 天
激进: 已完成 250/334 天
激进: 已完成 300/334 天
激进: 完成，用时 1.0s，求解器统计 {'HiGHS LP': 334}


,策略,date,计划购电量(kWh),紧急购电量(kWh),未利用电量(kWh),计划费用(元),紧急费用(元),总费用(元),期初储能(kWh),期末储能(kWh),求解器
0,保守,2025-02-01,64180.900084,0.0,23495.338314,36410.134861,0.0,36410.134861,6000.000000,9331.225353,HiGHS LP
1,保守,2025-02-02,96926.156099,0.0,16565.981951,63192.210745,0.0,63192.210745,9331.225353,7725.230805,HiGHS LP
2,保守,2025-02-03,92082.728311,0.0,8599.672382,59190.844633,0.0,59190.844633,7725.230805,7089.581254,HiGHS LP
3,保守,2025-02-04,96152.615091,0.0,10072.807963,62254.564456,0.0,62254.564456,7089.581254,7820.205064,HiGHS LP
4,保守,2025-02-05,95100.835259,0.0,11040.998873,61468.573082,0.0,61468.573082,7820.205064,7125.529022,HiGHS LP


## 5. 实际净负载的全评价期最优解

In [5]:
def solve_full_horizon_actual_optimum(actual_net, prices, cfg=CFG):
    """提前知道全评价期实际净负载时的统一最优解。"""
    net = np.asarray(actual_net, dtype=float).reshape(-1)
    p = np.tile(np.asarray(prices, dtype=float), len(actual_net))
    return solve_dispatch_for_net(
        net,
        p,
        initial_kwh=cfg["initial_storage_kwh"],
        terminal_kwh=cfg["terminal_storage_kwh"],
        cfg=cfg,
    )


started = time.perf_counter()
actual_opt = solve_full_horizon_actual_optimum(actual_net, prices)
print(f"实际最优求解完成，用时 {time.perf_counter() - started:.1f}s，求解器：{actual_opt['solver']}")

actual_opt_detail = pd.DataFrame({
    "date": np.repeat(dates, 144),
    "slot": np.tile(np.arange(1, 145), len(dates)),
    "actual_net_kwh": actual_net.reshape(-1),
    "planned_grid_kwh": actual_opt["grid"],
    "charge_kwh": actual_opt["charge"],
    "discharge_kwh": actual_opt["discharge"],
    "unused_kwh": actual_opt["unused"],
    "storage_start_kwh": actual_opt["storage"][:-1],
    "storage_end_kwh": actual_opt["storage"][1:],
    "price_yuan_per_kwh": np.tile(prices, len(dates)),
})

actual_opt_daily = actual_opt_detail.groupby("date", as_index=False).agg(
    **{
        "计划购电量(kWh)": ("planned_grid_kwh", "sum"),
        "紧急购电量(kWh)": ("actual_net_kwh", lambda x: 0.0),
        "未利用电量(kWh)": ("unused_kwh", "sum"),
        "期初储能(kWh)": ("storage_start_kwh", "first"),
        "期末储能(kWh)": ("storage_end_kwh", "last"),
    }
)
actual_opt_daily["策略"] = "实际最优"
actual_opt_daily["计划费用(元)"] = actual_opt_detail.eval("planned_grid_kwh * price_yuan_per_kwh").groupby(actual_opt_detail["date"]).sum().to_numpy()
actual_opt_daily["紧急费用(元)"] = 0.0
actual_opt_daily["总费用(元)"] = actual_opt_daily["计划费用(元)"]
actual_opt_daily["求解器"] = actual_opt["solver"]
actual_opt_daily = actual_opt_daily[daily_results.columns]

display(actual_opt_daily.head())

实际最优求解完成，用时 2.1s，求解器：HiGHS LP


,策略,date,计划购电量(kWh),紧急购电量(kWh),未利用电量(kWh),计划费用(元),紧急费用(元),总费用(元),期初储能(kWh),期末储能(kWh),求解器
0,实际最优,2025-02-01,40952.083487,0.0,0.0,20960.920907,0.0,20960.920907,6000.0,8550.0,HiGHS LP
1,实际最优,2025-02-02,82850.757560,0.0,0.0,51615.651662,0.0,51615.651662,8550.0,8550.0,HiGHS LP
2,实际最优,2025-02-03,84738.342098,0.0,0.0,52905.169604,0.0,52905.169604,8550.0,8550.0,HiGHS LP
3,实际最优,2025-02-04,85886.574585,0.0,0.0,53854.122418,0.0,53854.122418,8550.0,8550.0,HiGHS LP
4,实际最优,2025-02-05,85326.532036,0.0,0.0,53218.160077,0.0,53218.160077,8550.0,8550.0,HiGHS LP


## 6. 最后对比

In [6]:
all_daily = pd.concat([daily_results, actual_opt_daily], ignore_index=True)

summary = all_daily.groupby("策略", as_index=False).agg(
    **{
        "计划购电量(kWh)": ("计划购电量(kWh)", "sum"),
        "紧急购电量(kWh)": ("紧急购电量(kWh)", "sum"),
        "未利用电量(kWh)": ("未利用电量(kWh)", "sum"),
        "计划费用(元)": ("计划费用(元)", "sum"),
        "紧急费用(元)": ("紧急费用(元)", "sum"),
        "总费用(元)": ("总费用(元)", "sum"),
        "发生紧急购电天数": ("紧急购电量(kWh)", lambda s: int((s > 1e-6).sum())),
        "期初储能(kWh)": ("期初储能(kWh)", "first"),
        "期末储能(kWh)": ("期末储能(kWh)", "last"),
    }
)

order = pd.CategoricalDtype(["保守", "中庸", "激进", "实际最优"], ordered=True)
summary["策略"] = summary["策略"].astype(order)
summary = summary.sort_values("策略").reset_index(drop=True)
best_cost = float(summary.loc[summary["策略"].eq("实际最优"), "总费用(元)"].iloc[0])
summary["相对实际最优多花(元)"] = summary["总费用(元)"] - best_cost
summary["相对实际最优多花比例"] = summary["相对实际最优多花(元)"] / best_cost

display(
    summary.style.format({
        "计划购电量(kWh)": "{:,.2f}",
        "紧急购电量(kWh)": "{:,.2f}",
        "未利用电量(kWh)": "{:,.2f}",
        "计划费用(元)": "{:,.2f}",
        "紧急费用(元)": "{:,.2f}",
        "总费用(元)": "{:,.2f}",
        "期初储能(kWh)": "{:,.2f}",
        "期末储能(kWh)": "{:,.2f}",
        "相对实际最优多花(元)": "{:,.2f}",
        "相对实际最优多花比例": "{:.2%}",
    })
)

print("策略解释：")
print("保守使用 p95 净负载制定完整计划，普通购电和计划充电通常更多，紧急购电风险最低。")
print("中庸使用 p50 中心预测制定完整计划，在普通购电与紧急购电之间折中。")
print("激进使用 p05 净负载制定完整计划，普通购电少，但实际净负载偏高时会产生更多紧急购电。")
print("实际最优提前知道实际净负载，是评价期内统一优化得到的理想下界，不是可在线实施策略。")

,策略,计划购电量(kWh),紧急购电量(kWh),未利用电量(kWh),计划费用(元),紧急费用(元),总费用(元),发生紧急购电天数,期初储能(kWh),期末储能(kWh),相对实际最优多花(元),相对实际最优多花比例
0,保守,"23,180,766.55","12,690.66","4,094,368.43","14,323,159.64","74,886.72","14,398,046.36",56,"6,000.00","6,140.34","2,167,662.72",17.72%
1,中庸,"20,139,421.42","512,092.63","1,463,703.04","12,184,369.82","3,183,970.76","15,368,340.59",276,"6,000.00","5,794.36","3,137,956.94",25.66%
2,激进,"17,352,491.01","2,646,654.97","1,000,388.69","10,355,988.75","15,685,389.74","26,041,378.49",334,"6,000.00","5,303.65","13,810,994.84",112.92%
3,实际最优,"20,189,815.80",0.00,"986,180.69","12,230,383.65",0.00,"12,230,383.65",0,"6,000.00","6,000.00",0.00,0.00%


策略解释：
保守使用 p95 净负载，普通购电多，但紧急购电风险最低。
中庸使用 p50 中心预测，在普通购电与紧急购电之间折中。
激进使用 p05 净负载，普通购电少，但实际偏高时会产生更多紧急购电。
实际最优提前知道实际净负载，是评价期内统一优化得到的理想下界，不是可在线实施策略。
